
# Thí nghiệm 02 — Dự báo chuỗi thời gian khí hậu

## 1. Khảo sát dữ liệu tốc độ gió CMIP6

### 1.1. Mục tiêu

Notebook này thực hiện các bước chuẩn bị dữ liệu tốc độ gió gần bề mặt từ bộ dữ liệu khí hậu khu vực châu Âu.

Quy trình bao gồm:

1. Khảo sát cấu trúc và thuộc tính của tệp NetCDF.
2. Kiểm tra thông tin thời gian và tọa độ không gian.
3. Xác định các vị trí lấy mẫu trên khu vực Đức.
4. Trích xuất chuỗi tốc độ gió tại 5 vị trí.
5. Chuẩn hóa dữ liệu và tạo các cửa sổ dự báo.
6. Lưu dữ liệu đã xử lý để phục vụ các notebook tiếp theo.

### 1.2. Nguồn dữ liệu

Dữ liệu được đọc từ tệp:

`Klima_Daten_10m_3h_2020_RCP26.nc`

Theo metadata của tệp, dữ liệu có các đặc điểm:

| Thuộc tính | Giá trị |
|---|---|
| Dự án | CORDEX |
| Miền khí hậu | EUR-11 |
| Mô hình khí hậu khu vực | CLMcom-BTU-CCLM4-8-17 |
| Kịch bản | RCP2.6 |
| Năm dữ liệu | 2020 |
| Tần suất | 3 giờ |
| Biến sử dụng | `sfcWind` |
| Đại lượng | Tốc độ gió gần bề mặt |
| Đơn vị | m/s |
| Độ cao | 10 m |

Dữ liệu nguồn là kết quả mô phỏng khí hậu, không phải quan trắc trực tiếp tại các trạm thời tiết.

### 1.3. Cấu trúc dữ liệu

Tệp NetCDF chứa các chiều:

| Chiều | Kích thước |
|---|---:|
| `time` | 2928 |
| `rlat` | 412 |
| `rlon` | 424 |
| `vertices` | 4 |

Biến `sfcWind` có kích thước:

$$
(2928,\ 412,\ 424)
$$

Tương ứng với 2928 thời điểm và lưới không gian gồm 412 × 424 ô.

Vì dữ liệu được lưu trên lưới khí hậu xoay, notebook sử dụng tọa độ địa lý `lat` và `lon` để xác định vị trí lấy mẫu trên bản đồ thực tế.


In [ ]:
from pathlib import Path
import xarray as xr

from pathlib import Path
import xarray as xr

# 1. Tìm thư mục gốc của project
project_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "experiments" / "02_climate").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc của project."
    )

# 2. Xác định đường dẫn dữ liệu CMIP6
experiment_dir = project_root / "experiments" / "02_climate"

nc_path = (
    project_root
    / "datasets"
    / "climate"
    / "raw"
    / "Klima_Daten_10m_3h_2020_RCP26.nc"
)

if not nc_path.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy tệp NetCDF: {nc_path}"
    )

# 3. Mở tệp NetCDF
ds = xr.open_dataset(nc_path, decode_times=False)

# 4. Tổng quan dữ liệu
print("=== CMIP6 DATASET OVERVIEW ===")
print(ds)

# 5. Kiểm tra các chiều dữ liệu
print("\n=== DIMENSIONS ===")
for dim, size in ds.sizes.items():
    print(f"{dim}: {size}")

# 6. Kiểm tra hệ tọa độ
print("\n=== COORDINATES ===")
for name, coord in ds.coords.items():
    print(
        f"{name}: shape={coord.shape}, "
        f"dtype={coord.dtype}, "
        f"units={coord.attrs.get('units', 'N/A')}"
    )

# 7. Kiểm tra các biến khí hậu
print("\n=== DATA VARIABLES ===")
for name, var in ds.data_vars.items():
    print(f"\nVariable: {name}")
    print("Shape:", var.shape)
    print("Dimensions:", var.dims)
    print("Data type:", var.dtype)
    print("Long name:", var.attrs.get("long_name", "N/A"))
    print("Units:", var.attrs.get("units", "N/A"))

# 8. Kiểm tra metadata của tệp
print("\n=== GLOBAL ATTRIBUTES ===")
for key, value in ds.attrs.items():
    print(f"{key}: {value}")

ds.close()


=== CMIP6 DATASET OVERVIEW ===
<xarray.Dataset> Size: 2GB
Dimensions:       (rlon: 424, rlat: 412, time: 2928, vertices: 4)
Coordinates:
  * rlon          (rlon) float64 3kB -28.38 -28.26 -28.16 ... 17.93 18.05 18.16
  * rlat          (rlat) float64 3kB -23.38 -23.26 -23.16 ... 21.61 21.73 21.83
  * time          (time) float64 23kB 2.56e+04 2.56e+04 ... 2.596e+04 2.596e+04
    lat           (rlat, rlon) float64 1MB ...
    lon           (rlat, rlon) float64 1MB ...
    height        float64 8B ...
Dimensions without coordinates: vertices
Data variables:
    rotated_pole  |S1 1B ...
    sfcWind       (time, rlat, rlon) float32 2GB ...
    lon_vertices  (rlat, rlon, vertices) float64 6MB ...
    lat_vertices  (rlat, rlon, vertices) float64 6MB ...
Attributes: (12/23)
    contact:                        cordex-clm@dkrz.de, Klaus Keuler (keuler@...
    CORDEX_domain:                  EUR-11
    institute_id:                   CLMcom-BTU
    institution:                    Chair of Environ


## 2. Phân tích thông tin thời gian và không gian

### 2.1. Trục thời gian

Dữ liệu bao phủ toàn bộ năm 2020, từ:

- Thời điểm bắt đầu: 01/01/2020, 00:00.
- Thời điểm kết thúc: 31/12/2020, 21:00.

Khoảng thời gian giữa hai quan sát liên tiếp là 3 giờ.

Vì năm 2020 là năm nhuận nên tổng số quan sát là:

$$
366\times 8=2928
$$

Tọa độ thời gian trong tệp được mã hóa theo đơn vị:

`days since 1949-12-01 00:00:00`

Do đó, notebook sử dụng hàm `num2date` để chuyển giá trị thời gian số thành ngày giờ có thể đọc được.

### 2.2. Lưới không gian

Dữ liệu sử dụng hệ tọa độ xoay gồm hai chiều `rlat` và `rlon`.

Tệp đồng thời cung cấp hai biến tọa độ địa lý:

- `lat`: vĩ độ thực tế.
- `lon`: kinh độ thực tế.

Hai biến này có cùng kích thước:

$$
(412,\ 424)
$$

Tổng số điểm trên lưới:

$$
412\times 424=174688
$$

Mỗi phần tử của hai ma trận `lat` và `lon` xác định vị trí địa lý thực tế của một ô lưới.

### 2.3. Biến tốc độ gió

Biến `sfcWind` biểu diễn tốc độ gió ở độ cao 10 m so với bề mặt.

Dữ liệu có dạng:

$$
W\in\mathbb{R}^{2928\times412\times424}
$$

Trong đó $W_{t,i,j}$ là tốc độ gió tại thời điểm $t$ và vị trí lưới $(i,j)$.

Mục tiêu của bước tiếp theo là trích xuất chuỗi thời gian từ 5 vị trí đại diện trên khu vực Đức, thay vì sử dụng toàn bộ 174688 ô lưới.


In [ ]:
from netCDF4 import num2date

ds = xr.open_dataset(nc_path, decode_times=False)

# 1. Kiểm tra thông tin thời gian
time = ds["time"]
time_units = time.attrs["units"]
calendar = time.attrs.get("calendar", "standard")

dates = num2date(
    [time.values[0], time.values[-1]],
    units=time_units,
    calendar=calendar
)

print("=== TIME INFORMATION ===")
print("Start:", dates[0])
print("End:", dates[1])
print("Number of timestamps:", ds.sizes["time"])
print("Time interval (hours):", (time.values[1] - time.values[0]) * 24)

# 2. Kiểm tra thông tin không gian
lat = ds["lat"]
lon = ds["lon"]

print("\n=== SPATIAL INFORMATION ===")
print("Grid shape:", lat.shape)
print("Latitude range:", float(lat.min()), "to", float(lat.max()))
print("Longitude range:", float(lon.min()), "to", float(lon.max()))
print("Number of grid points:", ds.sizes["rlat"] * ds.sizes["rlon"])

# 3. Kiểm tra metadata của biến tốc độ gió
wind = ds["sfcWind"]

print("\n=== WIND SPEED ===")
print("Shape:", wind.shape)
print("Units:", wind.attrs.get("units"))
print("Height:", ds["height"].values.item())
print("Height units:", ds["height"].attrs.get("units"))

ds.close()


=== TIME INFORMATION ===
Start: 2020-01-01 00:00:00
End: 2020-12-31 21:00:00
Number of timestamps: 2928
Time interval (hours): 3.0

=== SPATIAL INFORMATION ===
Grid shape: (412, 424)
Latitude range: 21.987829208374023 to 72.58499908447266
Longitude range: -44.59386444091797 to 64.9643783569336
Number of grid points: 174688

=== WIND SPEED ===
Shape: (2928, 412, 424)
Units: m s-1
Height: 10.0
Height units: m



## 3. Xác định các vị trí lấy mẫu trên khu vực Đức

### 3.1. Mục tiêu

Bộ dữ liệu NetCDF sử dụng lưới tọa độ khí hậu xoay. Vì vậy, không thể trực tiếp sử dụng chỉ số hàng và cột để biểu diễn một vị trí địa lý trên lãnh thổ Đức.

Notebook xây dựng một tập các tọa độ mục tiêu, sau đó tìm ô lưới NetCDF gần nhất với từng tọa độ.

### 3.2. Xây dựng lưới tọa độ mục tiêu

Phạm vi tọa độ được lựa chọn:

| Thành phần | Giá trị |
|---|---|
| Vĩ độ | Từ 47.3° đến dưới 55.1° |
| Kinh độ | Từ 5.9° đến dưới 15.0° |
| Bước lấy mẫu | 0.05° |

Lưới được tạo bằng `numpy.arange()` và `numpy.meshgrid()`.

Thứ tự tạo và làm phẳng lưới được giữ nguyên nhằm bảo toàn ý nghĩa của các chỉ số vị trí đã sử dụng trong thí nghiệm.

### 3.3. Tìm điểm lưới gần nhất

Để tìm điểm gần nhất, tọa độ địa lý được chuyển sang biểu diễn Cartesian ba chiều trên mặt cầu đơn vị:

$$
x=\cos(\varphi)\cos(\lambda)
$$

$$
y=\cos(\varphi)\sin(\lambda)
$$

$$
z=\sin(\varphi)
$$

Trong đó:

- $\varphi$ là vĩ độ, tính bằng radian.
- $\lambda$ là kinh độ, tính bằng radian.

Sau đó sử dụng `scipy.spatial.cKDTree` để tìm điểm gần nhất trên lưới NetCDF đối với từng tọa độ mục tiêu.

Các chỉ số tìm được được chuyển ngược về chỉ số hàng `rlat` và cột `rlon`.

### 3.4. Lựa chọn 5 vị trí

Các chỉ số mục tiêu được sử dụng trong thí nghiệm:

`[0, 5000, 10000, 15000, 20000]`

Kết quả ánh xạ đã ghi nhận:

| Chỉ số | Tọa độ mục tiêu (lat, lon) | Tọa độ lưới gần nhất |
|---|---|---|
| 0 | (47.3000, 5.9000) | (47.2835, 5.9745) |
| 5000 | (53.9500, 7.4500) | (53.9985, 7.5121) |
| 10000 | (52.7500, 9.0500) | (52.7902, 9.0800) |
| 15000 | (51.5500, 10.6500) | (51.5630, 10.5647) |
| 20000 | (50.3500, 12.2500) | (50.3357, 12.3153) |

Đây là các vị trí được lựa chọn theo chỉ số trên lưới mục tiêu, không phải năm trạm quan trắc thực tế.

### 3.5. Vai trò trong thí nghiệm tiếp theo

Các chỉ số hàng và cột tìm được sẽ được lưu cùng dữ liệu tốc độ gió.

Thông tin này giúp xác định lại vị trí không gian khi xử lý các biến khí hậu khác, đặc biệt là Surface Pressure.

Vì vậy, cần bảo toàn thứ tự các vị trí và kết quả ánh xạ.


In [ ]:
import numpy as np
from scipy.spatial import cKDTree

# 1. Xây dựng lưới tọa độ mục tiêu trên khu vực Đức
lats = np.arange(47.3, 55.1, 0.05)
lons = np.arange(5.9, 15.0, 0.05)

target_lats, target_lons = np.meshgrid(lats, lons)
target_lats = target_lats.ravel()
target_lons = target_lons.ravel()


# 2. Chuyển tọa độ địa lý sang Cartesian ba chiều
def to_cartesian(lat, lon):
    lat = np.deg2rad(lat)
    lon = np.deg2rad(lon)

    return np.column_stack((
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ))


# 3. Tìm các điểm gần nhất trên lưới NetCDF
with xr.open_dataset(nc_path, decode_times=False) as ds:
    grid_lat = ds["lat"].values.ravel()
    grid_lon = ds["lon"].values.ravel()

    points = to_cartesian(grid_lat, grid_lon)
    targets = to_cartesian(target_lats, target_lons)

    tree = cKDTree(points)
    distances, grid_indices = tree.query(targets)

    source_i, source_j = np.unravel_index(
        grid_indices, ds["lat"].shape
    )


# 4. Lựa chọn 5 vị trí để trích xuất chuỗi thời gian
location_indices = [0, 5000, 10000, 15000, 20000]

print("=== CORRECTED LOCATION MAPPING ===")

for idx in location_indices:
    print(
        f"Location {idx:5d}: "
        f"Target=({target_lats[idx]:.4f}, {target_lons[idx]:.4f}), "
        f"Actual=({grid_lat[grid_indices[idx]]:.4f}, "
        f"{grid_lon[grid_indices[idx]]:.4f})"
    )

=== CORRECTED LOCATION MAPPING ===
Location     0: Target=(47.3000, 5.9000), Actual=(47.2835, 5.9745)
Location  5000: Target=(53.9500, 7.4500), Actual=(53.9985, 7.5121)
Location 10000: Target=(52.7500, 9.0500), Actual=(52.7902, 9.0800)
Location 15000: Target=(51.5500, 10.6500), Actual=(51.5630, 10.5647)
Location 20000: Target=(50.3500, 12.2500), Actual=(50.3357, 12.3153)


## 4. Trích xuất chuỗi tốc độ gió tại 5 vị trí

### 4.1. Mục tiêu

Sau khi xác định các ô lưới gần nhất, notebook trích xuất chuỗi thời gian của biến `sfcWind` tại từng vị trí.

Mỗi vị trí cung cấp một chuỗi gồm 2928 quan sát, tương ứng với toàn bộ năm 2020.

### 4.2. Phương pháp trích xuất

Với từng chỉ số vị trí, sử dụng cặp chỉ số hàng và cột đã tìm được:

- `source_i`: chỉ số trên chiều `rlat`.
- `source_j`: chỉ số trên chiều `rlon`.

Chuỗi thời gian được lấy bằng phép chọn:

`sfcWind.isel(rlat=i, rlon=j)`

Sau đó, năm chuỗi được ghép theo chiều cột để tạo thành ma trận dữ liệu.

### 4.3. Cấu trúc dữ liệu

Ma trận thu được:

$$
W\in\mathbb{R}^{2928\times5}
$$

Trong đó:

- Mỗi hàng là một thời điểm quan sát.
- Mỗi cột tương ứng với một vị trí không gian.
- Mỗi phần tử là tốc độ gió, đơn vị m/s.

### 4.4. Kiểm tra tính toàn vẹn

Kết quả kiểm tra từ lần trích xuất đã lưu:

| Thuộc tính | Giá trị |
|---|---|
| Kích thước | (2928, 5) |
| Kiểu dữ liệu | `float32` |
| Giá trị thiếu (NaN) | 0 |
| Giá trị vô cực (Inf) | 0 |
| Giá trị nhỏ nhất | 0.02670047 |
| Giá trị lớn nhất | 23.064035 |

Dữ liệu tại năm vị trí không chứa NaN hoặc Inf trong kết quả đã ghi nhận.

Ma trận này được lưu trong biến `wind_raw` và là nguồn dữ liệu cho các bước tiền xử lý tiếp theo.


In [ ]:
wind_series = []

# 1. Trích xuất chuỗi tốc độ gió tại 5 vị trí
with xr.open_dataset(nc_path, decode_times=False) as ds:
    for idx in location_indices:
        i = int(source_i[idx])
        j = int(source_j[idx])

        actual_lat = float(ds["lat"].isel(rlat=i, rlon=j))
        actual_lon = float(ds["lon"].isel(rlat=i, rlon=j))

        print(
            f"Location {idx:5d}: "
            f"Target=({target_lats[idx]:.4f}, {target_lons[idx]:.4f}), "
            f"Actual=({actual_lat:.4f}, {actual_lon:.4f})"
        )

        series = ds["sfcWind"].isel(
            rlat=i,
            rlon=j
        ).values.astype(np.float32)

        wind_series.append(series)

# 2. Ghép các chuỗi theo chiều cột
wind_raw = np.column_stack(wind_series)

# 3. Kiểm tra dữ liệu sau trích xuất
print("\n=== WIND SPEED DATA ===")
print("Shape:", wind_raw.shape)
print("Data type:", wind_raw.dtype)
print("Missing values:", np.isnan(wind_raw).sum())
print("Infinite values:", np.isinf(wind_raw).sum())
print("Minimum:", np.nanmin(wind_raw))
print("Maximum:", np.nanmax(wind_raw))

print("\nFirst 5 observations:")
print(wind_raw[:5])


Location     0: Target=(47.3000, 5.9000), Actual=(47.2835, 5.9745)
Location  5000: Target=(53.9500, 7.4500), Actual=(53.9985, 7.5121)
Location 10000: Target=(52.7500, 9.0500), Actual=(52.7902, 9.0800)
Location 15000: Target=(51.5500, 10.6500), Actual=(51.5630, 10.5647)
Location 20000: Target=(50.3500, 12.2500), Actual=(50.3357, 12.3153)

=== WIND SPEED DATA ===
Shape: (2928, 5)
Data type: float32
Missing values: 0
Infinite values: 0
Minimum: 0.02670047
Maximum: 23.064035

First 5 observations:
[[1.138077   4.102069   0.9765581  1.2122504  2.6071727 ]
 [0.44310927 2.7986221  0.9172874  0.864127   2.4495466 ]
 [1.300806   2.9371698  1.71782    1.2497466  2.7001398 ]
 [0.19893278 3.0117416  1.9249883  0.9192054  2.513055  ]
 [1.4373713  4.6838465  2.2188601  1.2293766  2.6096117 ]]



## 5. Tiền xử lý dữ liệu cho bài toán dự báo

### 5.1. Lựa chọn đoạn dữ liệu

Mặc dù ma trận `wind_raw` chứa 2928 thời điểm, bước chuẩn bị cửa sổ trong notebook này chỉ sử dụng 400 thời điểm đầu tiên:

$$
W_{\mathrm{sub}}\in\mathbb{R}^{400\times5}
$$

Mỗi thời điểm gồm năm giá trị tốc độ gió tại năm vị trí.

Lựa chọn này được giữ nguyên theo cấu hình thực nghiệm đã triển khai.

### 5.2. Chuẩn hóa Min-Max

Notebook sử dụng `MinMaxScaler` để chuẩn hóa từng kênh dữ liệu về khoảng từ 0 đến 1.

Công thức:

$$
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
$$

Với mỗi kênh, $x_{\min}$ và $x_{\max}$ được xác định trên đoạn 400 thời điểm đã chọn.

Sau chuẩn hóa, dữ liệu có giá trị nhỏ nhất bằng 0 và lớn nhất bằng 1.

**Lưu ý:** Trong nhánh thực nghiệm này, bộ chuẩn hóa được khớp trên toàn bộ 400 thời điểm được chọn. Notebook chưa thực hiện phân chia Train/Test độc lập ở bước này.

Quy trình chia Train/Test theo thời gian sẽ được xử lý riêng trong Scenario 4.

### 5.3. Tạo cửa sổ trượt

Cấu hình:

| Tham số | Giá trị |
|---|---:|
| Số thời điểm | 400 |
| Số kênh | 5 |
| Độ dài đầu vào (`patch_len`) | 80 |
| Chân trời dự báo (`horizon`) | 10 |
| Bước trượt | 1 |

Với vị trí bắt đầu $i$, dữ liệu đầu vào gồm 80 bước thời gian liên tiếp:

$$
X_i=W'[i:i+80]
$$

Nhãn dự báo gồm 10 bước tiếp theo:

$$
Y_i=W'[i+80:i+90]
$$

Các nhãn được chuyển vị và làm phẳng để tạo vector kích thước:

$$
5\times10=50
$$

Số cửa sổ được sinh ra:

$$
400-80-10=310
$$

### 5.4. Kết quả tiền xử lý

| Tensor | Kích thước | Ý nghĩa |
|---|---|---|
| `x_in` | (310, 80, 5) | Các chuỗi đầu vào |
| `x_out` | (310, 50) | Các giá trị cần dự báo |

Trong đó, thứ tự làm phẳng của nhãn được thực hiện theo từng kênh, mỗi kênh chứa 10 giá trị dự báo liên tiếp.

### 5.5. Liên hệ với Grid Search

Cấu hình `patch_len=80`, `horizon=10` là cấu hình minh họa được lưu trực tiếp trong tệp dữ liệu này.

Trong `03_climate_training.ipynb`, Grid Search sử dụng lại `wind_raw` để tạo các cửa sổ mới với nhiều độ dài đầu vào và chân trời dự báo khác nhau.

Vì vậy, không được hiểu rằng toàn bộ 120 lượt huấn luyện của Grid Search đều sử dụng cố định hai tensor `x_in` và `x_out` đang được tạo trong notebook này.


In [ ]:

import torch
from sklearn.preprocessing import MinMaxScaler

patch_len = 80
horizon = 10

# 1. Chọn 400 thời điểm đầu tiên
sub = wind_raw[:400, :]

# 2. Chuẩn hóa Min-Max theo từng kênh
scaler = MinMaxScaler().fit(sub)
sub = scaler.transform(sub)

# 3. Tạo các cửa sổ trượt
X, Y = [], []

for i in range(len(sub) - patch_len - horizon):
    X.append(sub[i:i + patch_len])
    y = sub[i + patch_len:i + patch_len + horizon].T.reshape(-1)
    Y.append(y)

# 4. Chuyển dữ liệu sang tensor PyTorch
x_in = torch.tensor(np.stack(X), dtype=torch.float32)
x_out = torch.tensor(np.stack(Y), dtype=torch.float32)

print("=== PREPROCESSED DATA ===")
print("Selected data:", sub.shape)
print("Normalized range:", sub.min(), sub.max())
print("X shape:", x_in.shape)
print("Y shape:", x_out.shape)
print("Number of samples:", len(x_in))


=== PREPROCESSED DATA ===
Selected data: (400, 5)
Normalized range: 0.0 1.0
X shape: torch.Size([310, 80, 5])
Y shape: torch.Size([310, 50])
Number of samples: 310



## 6. Lưu bộ dữ liệu đã xử lý

### 6.1. Mục tiêu

Sau khi hoàn thành trích xuất và tiền xử lý, dữ liệu được lưu dưới định dạng checkpoint PyTorch (`.pt`).

Tệp dữ liệu đầu ra:

`data/wind_nearest_5sites.pt`

Mục tiêu của việc lưu dữ liệu là tránh phải đọc lại tệp NetCDF dung lượng lớn và thực hiện lại các bước ánh xạ tọa độ, trích xuất và chuẩn hóa.

### 6.2. Nội dung được lưu

| Trường | Ý nghĩa |
|---|---|
| `wind_raw` | Ma trận tốc độ gió gốc tại 5 vị trí, kích thước (2928, 5) |
| `x_in` | Tensor đầu vào đã chuẩn hóa |
| `x_out` | Tensor nhãn dự báo |
| `location_indices` | Danh sách chỉ số vị trí được chọn |
| `source_i` | Chỉ số hàng trong lưới NetCDF |
| `source_j` | Chỉ số cột trong lưới NetCDF |
| `patch_len` | Độ dài chuỗi đầu vào |
| `horizon` | Số bước dự báo |
| `n_time_used` | Số thời điểm dùng để tạo cửa sổ |
| `sampling_method` | Phương pháp chọn vị trí gần nhất |
| `source_file` | Tên tệp NetCDF nguồn |

### 6.3. Vai trò đối với các notebook tiếp theo

Bộ dữ liệu đã lưu được sử dụng trong các bước phát triển và đánh giá mô hình dự báo khí hậu.

Trong đó:

- `02_climate_model.ipynb` sử dụng các tensor đã lưu để kiểm tra cấu hình dữ liệu đầu vào.
- `03_climate_training.ipynb` sử dụng `wind_raw` để xây dựng lại các cửa sổ tương ứng với từng cấu hình Grid Search.
- `04_climate_pressure_data.ipynb` sử dụng dữ liệu gió và thông tin vị trí để xác định các ô lưới tương ứng khi trích xuất Surface Pressure.

### 6.4. Bảo toàn dữ liệu

Tệp `wind_nearest_5sites.pt` đã được tạo từ lần thực nghiệm trước.

Trong quá trình tái cấu trúc repository, không tạo lại hoặc ghi đè tệp này.

Mã nguồn được giữ để mô tả đầy đủ phương pháp chuẩn bị dữ liệu và phục vụ khả năng tái lập trong tương lai.


In [ ]:
# 1. Xác định thư mục lưu dữ liệu
results_dir = experiment_dir / "data"
results_dir.mkdir(parents=True, exist_ok=True)

dataset_path = results_dir / "wind_nearest_5sites.pt"

torch.save({
    "wind_raw": torch.from_numpy(wind_raw.copy()),
    "x_in": x_in.cpu(),
    "x_out": x_out.cpu(),
    "location_indices": location_indices,
    "source_i": source_i[location_indices].tolist(),
    "source_j": source_j[location_indices].tolist(),
    "patch_len": patch_len,
    "horizon": horizon,
    "n_time_used": 400,
    "sampling_method": "nearest_neighbor",
    "source_file": Path(nc_path).name
}, dataset_path)

print("Dataset saved:", dataset_path)
print("X:", x_in.shape)
print("Y:", x_out.shape)


Dataset saved: C:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\wind_nearest_5sites.pt
X: torch.Size([310, 80, 5])
Y: torch.Size([310, 50])
